### Second Ticket
**TICKET**: *"Our nightly backup monitoring alert has been firing every morning for the last 3 days: 'backup file missing.' But when I asked the infra team if the backup job crashed, they said their dashboard shows it completing successfully every night. Something doesn't add up — can you dig in?"*

**SYSTEM**: Ubuntu 22.04 LTS (simulated). Backups run as a dedicated service account, `backupsvc`, writing into `/srv/backups`.

**START POINT**: logged in as `dennie`, `sudo` available.

---

1. Check backups that are written and should be served out from `/srv/backups`:

In [ ]:
dennie@HP:~$ ls -l /srv/backups
total 8
drwxr-xr-x 2 backupsvc backupsvc  4096 Aug 30 02:14 archives
lrwxrwxrwx 1 backupsvc backupsvc    30 Aug 26 02:15 latest -> archives/backup-2026-08-26.tar.gz

Read the symlink:

In [ ]:
dennie@HP:~$ readlink -f /srv/backups/latest
/srv/backups/archives/backup-2026-08-26.tar.gz

Check the target file exists:

In [ ]:
dennie@HP:~$ find /srv/backups -iname "backup-2026-08-26.tar.gz"
# (no output — no matches)

So **target file does NOT exist**.

Check archives `/srv/backups/archives`:

In [ ]:
dennie@HP:~$ ls -l /srv/backups/archives
total 1536000
---------- 1 backupsvc backupsvc 501234000 Aug 25 02:12 backup-2026-08-25.tar.gz.old
-rw-r----- 1 backupsvc backupsvc 498112000 Aug 28 02:14 backup-2026-08-28.tar.gz
-rw-r----- 1 backupsvc backupsvc 502345000 Aug 29 02:13 backup-2026-08-29.tar.gz
-rw-r----- 1 backupsvc backupsvc 499876000 Aug 30 02:14 backup-2026-08-30.tar.gz

Checking system logs:

In [ ]:
dennie@HP:~$ ls -l /var/log
total 2896
-rw-r----- 1 syslog     adm       12650 Aug 30 07:02 auth.log
drwxr-xr-x 2 root       root       4096 Feb 10  2026 apt
drwxr-xr-x 2 backupsvc  backupsvc  4096 Aug 30 02:14 backupjob
-rw-r--r-- 1 root       root     492454 Aug 30 07:02 dpkg.log
-rw-r----- 1 syslog     adm      124798 Aug 30 06:55 kern.log
drwxr-xr-x 2 monitor    monitor    4096 Aug 30 06:00 monitoring
-rw-r----- 1 syslog     adm      374713 Aug 30 07:02 syslog

In [ ]:
dennie@HP:~$ ls -l /var/log/monitoring
total 16
-rw-r--r-- 1 monitor monitor 14962 Aug 30 06:00 check-backup.log

**System logs** `/var/log/monitoring/check-backup.log` show, monitoring system logged some errors about checking backup files that are missing:

In [ ]:
dennie@HP:~$ grep -i "error\|warn" /var/log/monitoring/check-backup.log
2026-08-27 06:00:02 WARN Disk usage at 62% - within normal range
2026-08-28 06:00:01 ERROR cannot stat '/srv/backups/latest': No such file or directory
2026-08-28 06:00:01 ERROR Status: FAIL - backup missing or stale
2026-08-29 06:00:01 ERROR cannot stat '/srv/backups/latest': No such file or directory
2026-08-29 06:00:01 ERROR Status: FAIL - backup missing or stale
2026-08-30 06:00:01 ERROR cannot stat '/srv/backups/latest': No such file or directory
2026-08-30 06:00:01 ERROR Status: FAIL - backup missing or stale

Backup jobs have comleted successfully according **system logs** `/var/log/backupjob/backup.log`:

In [ ]:
dennie@HP:~$ ls -l /var/log/backupjob
total 44
-rw-r--r-- 1 backupsvc backupsvc 42318 Aug 30 02:14 backup.log

In [ ]:
dennie@HP:~$ cat /var/log/backupjob/backup.log
2026-08-24 02:00:01 INFO Starting backup job
2026-08-24 02:14:12 INFO Archive created: backup-2026-08-24.tar.gz size=497MB
2026-08-24 02:14:13 INFO Backup job completed successfully
2026-08-25 02:00:01 INFO Starting backup job
2026-08-25 02:14:20 INFO Archive created: backup-2026-08-25.tar.gz size=501MB
2026-08-25 02:14:21 INFO Backup job completed successfully
2026-08-26 02:00:01 INFO Starting backup job
2026-08-26 02:14:18 INFO Archive created: backup-2026-08-26.tar.gz size=499MB
2026-08-26 02:14:19 INFO Backup job completed successfully
2026-08-27 02:00:01 INFO Starting backup job
2026-08-27 02:14:09 INFO Archive created: backup-2026-08-27.tar.gz size=503MB
2026-08-27 02:14:10 INFO Backup job completed successfully
2026-08-28 02:00:01 INFO Starting backup job
2026-08-28 02:14:22 INFO Archive created: backup-2026-08-28.tar.gz size=498MB
2026-08-28 02:14:23 INFO Backup job completed successfully
2026-08-29 02:00:01 INFO Starting backup job
2026-08-29 02:13:58 INFO Archive created: backup-2026-08-29.tar.gz size=502MB
2026-08-29 02:13:59 INFO Backup job completed successfully
2026-08-30 02:00:01 INFO Starting backup job
2026-08-30 02:14:15 INFO Archive created: backup-2026-08-30.tar.gz size=499MB
2026-08-30 02:14:16 INFO Backup job completed successfully

checking other fies exist:

In [ ]:
find /srv/backups -iname "backup-2026-08-27.tar.gz"
# (no output - also gone)

Retention has purged both `08-26` and `08-27` by now, matching what you saw in `ls -l /srv/backups/archives` — only 28, 29, 30 remain.

In [ ]:
dennie@HP:~$ find /srv/backups -iname "backup-2026-08-28.tar.gz" && find /srv/backups -iname "backup-2026-08-29.tar.gz" && find /srv/backups -iname "backup-2026-08-30.tar.gz"
/srv/backups/archives/backup-2026-08-28.tar.gz
/srv/backups/archives/backup-2026-08-29.tar.gz
/srv/backups/archives/backup-2026-08-30.tar.gz

At this stage I found:
1) there is a symlink named 'latest' points to the NOT-existent backup file for 26-08 (`backup-2026-08-26.tar.gz`) while the latest one in the archive is for 30-08.
- I can fix it, to redirect this link to the actual last backup file. `ln -sfn /srv/backups/archives/backup-2026-08-30.tar.gz`
2) monitoring system has got several error logs about the backup file missing in `/var/log/monitoring/check-backup.log`, this confirms the ticket description about the main problem.
3) Earlier, in `/srv/backups/archives` I found those backup files, they are in place, but only for 30, 29, 28 and 25-08. The 'latest' symlink points to `latest -> archives/backup-2026-08-26.tar.gz` which does NOT exist.
4) The infra team's dashboard is fine since it reads the real backup files, `/var/log/backupjob/backup.log` probably what it logs out.

Your diagnosis (points 1–4) is spot on — broken symlink pointing at a purged archive:
- **monitoring correctly alerting on that**;
- **infra's dashboard reading a completely different (healthy) log** about the actual backup process.

That's the full root cause, correctly separated from the false lead of "backup job must be broken."

Fix the broken symlink:
`ln -s TARGET LINK_NAME` needs two arguments

In [ ]:
sudo ln -sfn /srv/backups/archives/backup-2026-08-30.tar.gz /srv/backups/latest
# (no output — success)

Verify the link:

In [ ]:
dennie@HP:~$ sudo ln -sfn /srv/backups/archives/backup-2026-08-30.tar.gz /srv/backups/latest
[sudo] password for dennie: 
dennie@HP:~$ readlink -f /srv/backups/latest
/srv/backups/archives/backup-2026-08-30.tar.gz

> 🔁 **Support rule**: for any command that changes state (`ln`, `chmod`, `chown`, `rm`), always double-check the full argument list before hitting enter — especially `ln -s`, where a missing second argument doesn't fail loudly, it just does something quietly different from what you intended.

## DEBRIEF

**Root cause:** the daily job that's supposed to repoint `latest` at each night's new archive had been silently broken since around `08-26` — the symlink never moved again after that. It stayed pointing at a real file for a couple more days (so nothing looked wrong yet), until nightly retention cleanup eventually purged the `08-26` archive itself, at which point the stale symlink became a genuinely _broken_ one — and that's the exact moment the monitoring alerts started, 3 mornings ago.

**The key skill this ticket tested that the first one didn't:** trusting **evidence over assumption**. The ticket description said "backup file missing," which invites the assumption "the backup job is broken." You didn't take that at face value — you pulled _both_ logs (`backupjob/backup.log`, showing a perfectly healthy job every night, and `monitoring/check-backup.log`, showing the actual failing check) and let the evidence tell you these were two different systems, only one of which was actually broken. That's a materially harder, more realistic skill than ticket one, where there was only one log to read.

**Echoes from earlier in the session that paid off here:**

- The `ln -s TARGET LINK_NAME` argument-order lesson from way back — caught yourself missing the second argument before it silently created a stray file in your home directory
- The "prompt text isn't part of the command" lesson — caught it happening again mid-chain, and understood _why_ only one command in the chain errored (the `&&` short-circuit)
- The Pillar 3 least-privilege/ownership model — correctly reasoned through why `dennie` needed `sudo` for a directory owned by a service account, even for something as small as fixing one symlink